In [1]:
import librosa
#from skimage import feature
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
from scipy import signal
import soundfile as sf

In [2]:
import pandas as pd

# --------------------------------------------------
# File paths
# --------------------------------------------------

main_file = "xc_FVA.csv"


# --------------------------------------------------
# Load files
# --------------------------------------------------

df = pd.read_csv(main_file)
 

In [3]:
df.drop(
    columns=["annotation-set", "sono", "osci", "url"],
    inplace=True
)




In [4]:
# Basic dataframe analysis

# Summary statistics for every column
summary = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "missing": df.isna().sum(),
    "unique_values": df.nunique(dropna=False),
    "missing_percent": (df.isna().mean() * 100).round(2)
})

display(summary)

# Number of duplicate rows
print(f"Duplicate rows: {df.duplicated().sum()}")

# Unique-value counts for each column
for column in df.columns:
    print(f"\n{'=' * 60}\n{column}")
    print(f"Unique values: {df[column].nunique(dropna=False)}")
    display(df[column].value_counts(dropna=False).head(20))

# Date analysis
dates = pd.to_datetime(df["date"], errors="coerce")
print("\nDate range:")
print(f"Earliest: {dates.min()}")
print(f"Latest:   {dates.max()}")

# Duration analysis
duration_seconds = pd.to_timedelta(
    "00:" + df["length"].astype(str),
    errors="coerce"
).dt.total_seconds()

print("\nDuration statistics in seconds:")
display(duration_seconds.describe())

# Numeric summary
print("\nNumeric columns:")
display(df.describe(include="number"))

# Categorical summary
print("\nCategorical columns:")
display(df.describe(include="object").T)

,dtype,non_null,missing,unique_values,missing_percent
id,int64,759,0,759,0.00
gen,object,759,0,103,0.00
sp,object,759,0,127,0.00
ssp,object,116,643,38,84.72
grp,object,759,0,2,0.00
status,object,759,0,1,0.00
en,object,759,0,134,0.00
rec,object,759,0,1,0.00
cnt,object,759,0,2,0.00
loc,object,759,0,6,0.00


Duplicate rows: 0

id
Unique values: 759


1025871    1
973123     1
963723     1
961336     1
959575     1
987548     1
987547     1
987544     1
965373     1
964791     1
963805     1
973122     1
987546     1
973121     1
973120     1
973119     1
973117     1
973116     1
973113     1
973112     1
Name: id, dtype: int64


gen
Unique values: 103


Turdus          42
Tringa          32
Anthus          27
Fringilla       24
Vanellus        23
Sylvia          22
Nycticorax      21
Emberiza        19
Rallus          18
Picus           18
Phylloscopus    17
Motacilla       15
Scolopax        15
Periparus       15
Dryocopus       15
Fulica          14
Chloris         14
Galerida        14
Iduna           13
Cyanistes       13
Name: gen, dtype: int64


sp
Unique values: 127


... (67 outputs left out)


Numeric columns:


,id,lat,lon,alt,temp,regnr,smp
count,7.590000e+02,745.000000,745.000000,759.000000,0.0,0.0,759.000000
mean,1.003917e+06,47.148925,10.568926,549.591568,NaN,NaN,46992.885375
std,6.158960e+04,2.586320,3.625669,258.227093,NaN,NaN,1708.013698
min,9.372600e+05,40.282600,8.078000,130.000000,NaN,NaN,44100.000000
25%,9.637900e+05,47.637500,8.145400,420.000000,NaN,NaN,44100.000000
50%,9.655240e+05,48.197500,9.756300,500.000000,NaN,NaN,48000.000000
75%,1.024660e+06,48.210200,9.756300,900.000000,NaN,NaN,48000.000000
max,1.154877e+06,48.519900,20.050300,1000.000000,NaN,NaN,48000.000000



Categorical columns:


,count,unique,top,freq
gen,759,103,Turdus,42
sp,759,127,vanellus,23
ssp,116,37,coelebs,9
grp,759,2,birds,748
status,759,1,identified,759
en,759,134,Northern Lapwing,23
rec,759,1,Working group for animal ecology and planning ...,759
cnt,759,2,Germany,653
loc,759,6,"Gedüngtes Ried, Schemmerhofen, Biberach, Baden...",349
type,754,20,call,360


Group by User and value_count mic field

In [5]:
mic_counts_by_rec = (
    df.groupby("rec")["mic"]
      .value_counts(dropna=False)
      .rename("count")
      .reset_index()
)

display(mic_counts_by_rec)

,rec,mic,count
0,Working group for animal ecology and planning ...,Audiomoth 1.2.0,759


In [6]:
count_audiomoth = df["file-name"].str.contains(
    "audiomoth",
    case=False,
    na=False
).sum()

print(count_audiomoth)

353


In [7]:
count_birdnet = df["file-name"].str.contains(
    "birdnet",
    case=False,
    na=False
).sum()

print(count_birdnet)

198


In [8]:
count_albania = (df["cnt"] == "Albania").sum()
print(count_albania)

106


In [9]:
count_wav = df["file-name"].str.contains(
    "mp3",
    case=False,
    na=False,
).sum()

print(count_wav)

207


In [11]:
df = df.loc[df["gen"] != "Scolopax"].copy()
df.to_csv("FVA_nonEur.csv", index=False)
print(f"Saved {len(df)} rows to Germany.csv")

Saved 744 rows to Germany.csv


In [12]:
total_duration_seconds = duration_seconds.sum()

print(f"Total duration (all rows): {total_duration_seconds:.2f} seconds")

Total duration (all rows): 35255.00 seconds
